Method 1: RF + CRF

Modular pipeline

In [ ]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
import sklearn_crfsuite
from nltk.tokenize.treebank import TreebankWordDetokenizer

ROOT_DIR = Path(r'C:\Users\Lenovo\Desktop\NLP project')
INPUT_CSV = ROOT_DIR / "outputs" / "super_full_ebm_data.csv"
OUTPUT_DIR = ROOT_DIR / "outputs"
detokenizer = TreebankWordDetokenizer()

def safe_eval(val):
    """Safely convert string representation of lists back to Python lists."""
    if isinstance(val, list): return val
    try:
        return ast.literal_eval(val)
    except:
        return []

FEATURE ENGINEERING FOR CRF

In [ ]:
def word2features(sent_tokens, i):
    word = str(sent_tokens[i])
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        features['-1:word.lower()'] = str(sent_tokens[i-1]).lower()
    else:
        features['BOS'] = True
    if i < len(sent_tokens) - 1:
        features['+1:word.lower()'] = str(sent_tokens[i+1]).lower()
    else:
        features['EOS'] = True
    return features


EXECUTION PIPELINE

In [ ]:
def run_axis2b_pipeline():
    print(">>> [Step 1/5] Loading preprocessed ground truth data...")
    df = pd.read_csv(INPUT_CSV)
    df['sentence_tokens'] = df['sentence_tokens'].apply(safe_eval)
    
    train_df = df[df['split'] == 'train'].copy()
    test_df = df[df['split'] == 'test'].copy()
    
    print(">>> [Step 2/5] Initializing Sentence Transformer...")
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    final_output = pd.DataFrame({'doc_id': test_df['doc_id'].unique()})
    
    entity_configs = [
        {'col': 'p_count', 'pred': 'participants_pred', 'name': 'Participants'},
        {'col': 'i_count', 'pred': 'interventions_pred', 'name': 'Interventions'},
        {'col': 'o_count', 'pred': 'outcomes_pred', 'name': 'Outcomes'}
    ]

    for config in entity_configs:
        entity_name = config['name']
        target_col = config['col']
        result_col = config['pred']
        
        print(f"\n--- Processing Entity Category: {entity_name} ---")
        
        print(f"Training Stage 1 (RF Classifier) for {entity_name}...")
        y_train_s1 = (train_df[target_col] > 0).astype(int)
        X_train_s1 = embed_model.encode(train_df['sentence_text'].tolist(), show_progress_bar=True)
        
        rf_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
        rf_clf.fit(X_train_s1, y_train_s1)
        
        print(f"Training Stage 2 (CRF Extractor) for {entity_name}...")
        crf_train_subset = train_df[train_df[target_col] > 0]
        X_crf = [[word2features(row['sentence_tokens'], i) for i in range(len(row['sentence_tokens']))] 
                 for _, row in crf_train_subset.iterrows()]
        y_crf = [['I' if int(target_col == 'p_count' and row['p_count'] > 0) or 
                      (target_col == 'i_count' and row['i_count'] > 0) or 
                      (target_col == 'o_count' and row['o_count'] > 0) else 'O' 
                  for _ in row['sentence_tokens']] for _, row in crf_train_subset.iterrows()]
        
        crf_model = sklearn_crfsuite.CRF(algorithm='lbfgs', max_iterations=100, all_possible_transitions=True)
        crf_model.fit(X_crf, y_crf)
        
        print(f"Predicting {entity_name} in Test Set...")
        X_test_s1 = embed_model.encode(test_df['sentence_text'].tolist(), show_progress_bar=True)
        test_df['s1_prediction'] = rf_clf.predict(X_test_s1)
        
        doc_results = []
        for d_id, group in test_df.groupby('doc_id'):
            found_phrases = set()
            for _, row in group[group['s1_prediction'] == 1].iterrows():
                feats = [word2features(row['sentence_tokens'], i) for i in range(len(row['sentence_tokens']))]
                tags = crf_model.predict_single(feats)
                
                current_phrase = []
                for i, tag in enumerate(tags):
                    if tag == 'I':
                        current_phrase.append(row['sentence_tokens'][i])
                    else:
                        if current_phrase:
                            txt = detokenizer.detokenize(current_phrase).strip()
                            if any(c.isalpha() for c in txt): found_phrases.add(txt)
                            current_phrase = []
                if current_phrase:
                    txt = detokenizer.detokenize(current_phrase).strip()
                    if any(c.isalpha() for c in txt): found_phrases.add(txt)
            
            doc_results.append({'doc_id': d_id, result_col: "; ".join(sorted(list(found_phrases)))})
        
        entity_df = pd.DataFrame(doc_results)
        final_output = final_output.merge(entity_df, on='doc_id', how='left')

    final_output = final_output.fillna("")
    save_path = OUTPUT_DIR / "AITA_CW_Grp37_Axis2B_FINAL_RESULTS.csv"
    final_output.to_csv(save_path, index=False)
    print(f"\n✨ SUCCESS! Pipeline complete. Final report saved to: {save_path}")

if __name__ == "__main__":
    run_axis2b_pipeline()

Extraction and merge

In [ ]:
TARGET_ENTITY = 'outcomes'  

ROOT_DIR = Path(r'C:\Users\Lenovo\Desktop\NLP project')
DATA_DIR = ROOT_DIR / "ebm_nlp_2_00"
OUTPUT_DIR = ROOT_DIR / "outputs"
detokenizer = TreebankWordDetokenizer()

In [ ]:
def word2features(sent_tokens, i):
    word = str(sent_tokens[i])
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        features['-1:word.lower()'] = str(sent_tokens[i-1]).lower()
    else:
        features['BOS'] = True
    if i < len(sent_tokens) - 1:
        features['+1:word.lower()'] = str(sent_tokens[i+1]).lower()
    else:
        features['EOS'] = True
    return features

def build_task_data(entity_type):
    print(f"\n>>> [Step 1/5] ")
    all_rows = []
    for split in ["train", "test"]:
        subdir = "test/gold" if split == "test" else "train"
        id_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / entity_type / subdir
        doc_ids = sorted([p.name.split('.')[0] for p in id_path.glob("*.ann")])
        
        for d_id in doc_ids:
            with open(DATA_DIR / "documents" / f"{d_id}.tokens", "r", encoding="utf-8") as f:
                tokens = [line.strip() for line in f]
            
            ann_file = id_path / f"{d_id}.AGGREGATED.ann"
            if not ann_file.exists(): ann_file = id_path / f"{d_id}.ann"
            with open(ann_file, "r", encoding="utf-8") as f:
                labels_raw = [line.strip() for line in f]

            start = 0
            for s_idx, token in enumerate(tokens):
                if token in {".", "!", "?"} or s_idx == len(tokens) - 1:
                    end = s_idx + 1
                    s_tokens = tokens[start:end]
                    s_bio = ['I' if l != '0' else 'O' for l in labels_raw[start:end]]
                    
                    all_rows.append({
                        "doc_id": d_id,
                        "split": split,
                        "sentence_text": detokenizer.detokenize(s_tokens),
                        "sentence_tokens": s_tokens,
                        "bio_labels": s_bio,
                        "has_target": 1 if 'I' in s_bio else 0
                    })
                    start = end
    return pd.DataFrame(all_rows)

df_all = build_task_data(TARGET_ENTITY)
train_df = df_all[df_all['split'] == 'train']
test_df = df_all[df_all['split'] == 'test']

Stage 1: classification

In [ ]:
print(f">>> [Step 2/5] ")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
X_train_s1 = embed_model.encode(train_df['sentence_text'].tolist(), show_progress_bar=True)
rf_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_clf.fit(X_train_s1, train_df['has_target'])

Stage 2: CRF

In [ ]:
print(f">>> [Step 3/5] ")
crf_train = train_df[train_df['has_target'] == 1]
X_crf = [[word2features(row['sentence_tokens'], i) for i in range(len(row['sentence_tokens']))] for _, row in crf_train.iterrows()]
y_crf = [row['bio_labels'] for _, row in crf_train.iterrows()]
crf_model = sklearn_crfsuite.CRF(algorithm='lbfgs', max_iterations=100, all_possible_transitions=True)
crf_model.fit(X_crf, y_crf)

print(f">>> [Step 4/5] ")
X_test_s1 = embed_model.encode(test_df['sentence_text'].tolist(), show_progress_bar=True)
test_df = test_df.copy()
test_df['pred_active'] = rf_clf.predict(X_test_s1)

print(f">>> [Step 5/5] ")
entity_results = []
for d_id, group in test_df.groupby('doc_id'):
    extracted = set()
    for _, row in group[group['pred_active'] == 1].iterrows():
        feats = [word2features(row['sentence_tokens'], i) for i in range(len(row['sentence_tokens']))]
        tags = crf_model.predict_single(feats)
        
        curr_phrase = []
        for i, tag in enumerate(tags):
            if tag == 'I':
                curr_phrase.append(row['sentence_tokens'][i])
            else:
                if curr_phrase:
                    txt = detokenizer.detokenize(curr_phrase).strip()
                    if any(c.isalpha() for c in txt) and len(txt) > 1:
                        extracted.add(txt)
                    curr_phrase = []
        if curr_phrase:
            txt = detokenizer.detokenize(curr_phrase).strip()
            if any(c.isalpha() for c in txt) and len(txt) > 1:
                extracted.add(txt)
    
    entity_results.append({
        "doc_id": d_id,
        f"{TARGET_ENTITY}_pred": "; ".join(sorted(list(extracted)))
    })

temp_file = OUTPUT_DIR / f"temp_{TARGET_ENTITY}_results.csv"
pd.DataFrame(entity_results).to_csv(temp_file, index=False)

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(r'C:\Users\Lenovo\Desktop\NLP project\outputs')

df_i = pd.read_csv(OUTPUT_DIR / "AITA_CW_Grp37_Axis2B_Results_Final.csv")
df_i = df_i.drop(columns=['participants_pred', 'outcomes_pred'], errors='ignore')

df_p = pd.read_csv(OUTPUT_DIR / "temp_participants_results.csv")
df_o = pd.read_csv(OUTPUT_DIR / "temp_outcomes_results.csv")

final_df = df_i.merge(df_p, on='doc_id', how='outer').merge(df_o, on='doc_id', how='outer')

final_df = final_df[['doc_id', 'participants_pred', 'interventions_pred', 'outcomes_pred']]
final_df = final_df.fillna("") 
final_df = final_df.sort_values('doc_id') 

final_submission_path = OUTPUT_DIR / "AITA_CW_Grp37_Axis2B_FULL_SUBMISSION.csv"
final_df.to_csv(final_submission_path, index=False)

print(final_df.head())

Method 2: PubMedBERT + CRF

Part 1: Environment Setup and Path Configuration

In [ ]:
import pandas as pd
import numpy as np
import ast
import os
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
import sklearn_crfsuite
from nltk.tokenize.treebank import TreebankWordDetokenizer

BASE_PATH = Path(r'C:\Users\Lenovo\Desktop\NLP project\ebm_nlp_2_00')
INPUT_CSV = BASE_PATH / "axis2_B" / "outputs" / "sentence_records_df.csv"
EMBEDDINGS_PATH = BASE_PATH / "axis2_B" / "data" / "pubmedbert_embeddings.npy"
GOLD_DIR = BASE_PATH / "annotations" / "aggregated" / "hierarchical_labels" / "participants" / "test" / "gold"
OUTPUT_DIR = BASE_PATH / "axis2_B" / "outputs"

def safe_eval(val):
    if isinstance(val, list): return val
    try: return ast.literal_eval(val)
    except: return []

all_df = pd.read_csv(INPUT_CSV)
all_df['sentence_tokens'] = all_df['sentence_tokens'].apply(safe_eval)

sentence_embeddings = np.load(EMBEDDINGS_PATH)

official_test_ids = [f.split('.')[0] for f in os.listdir(GOLD_DIR) if f.endswith('.ann')]
official_test_ids = [int(i) for i in official_test_ids]

test_df = all_df[all_df['doc_id'].isin(official_test_ids)].copy()
train_df = all_df[~all_df['doc_id'].isin(official_test_ids)].copy()

X_test_embeddings = sentence_embeddings[test_df.index]
X_train_embeddings = sentence_embeddings[train_df.index]

print(f"Total Sentences: {len(all_df)}")
print(f"Official Test Docs Found: {len(test_df['doc_id'].unique())}")
print(f"Training Sentences: {len(train_df)}")
print(f"Testing Sentences: {len(test_df)}")

Step 2: Sentence Embedding (PubMedBERT)

In [ ]:
embed_model = SentenceTransformer('microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext')

X_train_embeddings = embed_model.encode(train_df['sentence_text'].tolist(), show_progress_bar=True)
X_test_embeddings = embed_model.encode(test_df['sentence_text'].tolist(), show_progress_bar=True)

Step 3: CRF Feature Engineering

In [ ]:
def word2features(sent_tokens, i):
    word = str(sent_tokens[i])
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word.isupper()': word.isupper(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        features['-1:word.lower()'] = str(sent_tokens[i-1]).lower()
        if i > 1:
            features['-2:word.lower()'] = str(sent_tokens[i-2]).lower()
    else:
        features['BOS'] = True
    if i < len(sent_tokens) - 1:
        features['+1:word.lower()'] = str(sent_tokens[i+1]).lower()
        if i < len(sent_tokens) - 2:
            features['+2:word.lower()'] = str(sent_tokens[i+2]).lower()
    else:
        features['EOS'] = True
    return features

import numpy as np
import os
from pathlib import Path

sentence_embeddings = np.concatenate([X_train_embeddings, X_test_embeddings])

EMBEDDINGS_PATH = Path(r'C:\Users\Lenovo\Desktop\NLP project\ebm_nlp_2_00\axis2_B\data\pubmedbert_embeddings.npy')
EMBEDDINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(EMBEDDINGS_PATH, sentence_embeddings)

GOLD_DIR = Path(r'C:\Users\Lenovo\Desktop\NLP project\ebm_nlp_2_00\annotations\aggregated\hierarchical_labels\participants\test\gold')
official_test_ids = [int(f.split('.')[0]) for f in os.listdir(GOLD_DIR) if f.endswith('.ann')]

test_df = all_df[all_df['doc_id'].isin(official_test_ids)].copy()
train_df = all_df[~all_df['doc_id'].isin(official_test_ids)].copy()

X_test_embeddings = sentence_embeddings[test_df.index]
X_train_embeddings = sentence_embeddings[train_df.index]

print(f"Test docs matched: {len(test_df['doc_id'].unique())}")

Step 4: Hybrid Model Training and Prediction

In [ ]:
import os
import pandas as pd
import numpy as np
import nltk
import re
import sklearn_crfsuite
from sklearn.ensemble import RandomForestClassifier
from nltk.tokenize.treebank import TreebankWordDetokenizer
from pathlib import Path

def word2features_v19(sent, i, pos):
    w = str(sent[i])
    t = pos[i][1]
    med_suffixes = ('ine', 'one', 'ide', 'ate', 'pam', 'lol', 'vir', 'mab', 'tin', 'mic', 'cin')
    
    f = {
        'bias': 1.0,
        'w.lower()': w.lower(),
        'w.isupper()': w.isupper(),
        'w.isdigit()': w.isdigit(), 
        'w.med_suffix': w.lower().endswith(med_suffixes),
        'pos': t,
        'pos2': t[:2]
    }
    if i > 0:
        f.update({'-1:w.lower()': str(sent[i-1]).lower(), '-1:pos': pos[i-1][1]})
    else:
        f['BOS'] = True
    if i < len(sent) - 1:
        f.update({'+1:w.lower()': str(sent[i+1]).lower(), '+1:pos': pos[i+1][1]})
    else:
        f['EOS'] = True
    return f

def synthesize_labels_v19(pos, t_type):
    anchors = {
        'P': {'patients', 'men', 'women', 'adults', 'children', 'subjects', 'volunteers', 'infants', 'elderly', 'cases', 'smokers'},
        'I': {'mg', 'dose', 'treatment', 'therapy', 'tablet', 'drug', 'injection', 'infusion', 'placebo', 'capsule', 'intervention'},
        'O': {'rate', 'score', 'ratio', 'survival', 'incidence', 'primary', 'secondary', 'outcome', 'change', 'mortality', 'improvement'}
    }
    target = anchors[t_type]
    labels = ['O'] * len(pos)
    
    for i, (w, t) in enumerate(pos):
        if w.lower() in target:
            for j in range(max(0, i-2), min(len(pos), i+3)):
                if pos[j][1].startswith(('NN', 'JJ', 'CD')):
                    labels[j] = 'I'
    return labels

def clean_v19(entities, t_type):
    hdr = r'^(CONCLUSION|METHODS|RESULTS|BACKGROUND|DESIGN|OBJECTIVE|SETTING|AIM|POPULATION|STATISTICAL|STUDY)\s*[:\-]?\s*'
    frg = {'of', 'and', 'the', 'next', 'more', 'than', 'need', 'for', 'with', 'between', 'after', 'from', 'at', 'in', 'to', 'on'}
    
    res = []
    detok = TreebankWordDetokenizer()
    for e in entities:
        e = re.sub(hdr, '', e, flags=re.IGNORECASE).strip()
        tokens = e.split()
        if len(tokens) < 1 or len(tokens) > 5 or e.isupper(): continue
        if tokens[0].lower() in frg or tokens[-1].lower() in frg: continue
        
        f = re.sub(r'^(the|a|an|with|for|of|in|at|to|on|and)\s+', '', e, flags=re.IGNORECASE).strip()
        if len(f) > 2: res.append(f)
            
    final = []
    for x in sorted(list(set(res)), key=len, reverse=True):
        if not any(x.lower() in u.lower() and x.lower() != u.lower() for u in final):
            final.append(x)
    return "; ".join(final[:3]) 

detok = TreebankWordDetokenizer()
test_ids = test_df['doc_id'].unique()
final_output = pd.DataFrame({'doc_id': [str(i) for i in test_ids]})

if 'pos_tags' not in train_df.columns: train_df['pos_tags'] = train_df['sentence_tokens'].apply(nltk.pos_tag)
if 'pos_tags' not in test_df.columns: test_df['pos_tags'] = test_df['sentence_tokens'].apply(nltk.pos_tag)

configs = [
    {'col': 'p_count', 'target': 'participants_pred', 'type': 'P', 'rf_lim': 0.40},
    {'col': 'i_count', 'target': 'interventions_pred', 'type': 'I', 'rf_lim': 0.50},
    {'col': 'o_count', 'target': 'outcomes_pred', 'type': 'O', 'rf_lim': 0.50}
]

for cfg in configs:
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42).fit(X_train_embeddings, (train_df[cfg['col']] > 0).astype(int))
    sub = train_df[train_df[cfg['col']] > 0]
    crf = sklearn_crfsuite.CRF(algorithm='lbfgs', c1=0.1, c2=0.1, max_iterations=60).fit(
        [[word2features_v19(r['sentence_tokens'], j, r['pos_tags']) for j in range(len(r['sentence_tokens']))] for _, r in sub.iterrows()],
        [synthesize_labels_v19(r['pos_tags'], cfg['type']) for _, r in sub.iterrows()])
    
    test_df['prob'] = rf.predict_proba(X_test_embeddings)[:, 1]
    col_results = []
    for d_id, gp in test_df.groupby('doc_id'):
        raw = []
        for _, row in gp[gp['prob'] >= cfg['rf_lim']].nlargest(3, 'prob').iterrows():
            preds = crf.predict_single([word2features_v19(row['sentence_tokens'], j, row['pos_tags']) for j in range(len(row['sentence_tokens']))])
            curr = []
            for word, label in zip(row['sentence_tokens'], preds):
                if label == 'I': curr.append(word)
                elif curr: raw.append(detok.detokenize(curr)); curr = []
            if curr: raw.append(detok.detokenize(curr))
        col_results.append({'doc_id': str(d_id), cfg['target']: clean_v19(raw, cfg['type'])})
    
    final_output = final_output.merge(pd.DataFrame(col_results), on='doc_id', how='left')

final_output.fillna("").to_csv(OUTPUT_DIR / "Axis2B_RESULTS2.csv", index=False)